# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mimicodegirl-26/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My lane is content refresh prioritization. I frame it as a scoring and ranking problem because the goal is to order content items by how strongly they match the observed decline proxy and decide which pages deserve human review first. The output supports editorial decision-making; it does not guarantee that an update will improve performance.

In [4]:
# Section 1: Record and check the task framing

LANE = "Content refresh prioritization"
TASK_TYPE = "Scoring / ranking"
UNIT_OF_ANALYSIS = "One pseudonymized content item"
BUSINESS_ACTION = "Prioritize pages for human review"

assert TASK_TYPE in {"Classification", "Clustering", "Ranking", "Scoring",
                     "Scoring / ranking"}

print("Lane:", LANE)
print("Task type:", TASK_TYPE)
print("Unit of analysis:", UNIT_OF_ANALYSIS)
print("Action supported:", BUSINESS_ACTION)

Lane: Content refresh prioritization
Task type: Scoring / ranking
Unit of analysis: One pseudonymized content item
Action supported: Prioritize pages for human review


## 2. Target or proxy
The starter proxy target is is_declining_label, defined as 1 when trend_direction is "down" and 0 otherwise. This is a rule-derived label describing the observed trend category, not an independently measured future outcome. It is useful for learning the initial workflow, but a stronger future-facing task would predict decline over a later period using features from an earlier period. I will not use trend_direction or trend_pct as model features because they reveal the current proxy label.

In [5]:
# Section 2: Document the target and its limitations

TARGET_COLUMN = "is_declining_label"
SOURCE_COLUMN = "trend_direction"
LEAKAGE_COLUMNS = ["trend_direction", "trend_pct"]

LABEL_TYPE = "Rule-derived observed-trend proxy"
IDEAL_FUTURE_TARGET = (
    "Decline during a future time window, predicted from earlier data"
)

print("Target:", TARGET_COLUMN)
print("Label source:", f'{SOURCE_COLUMN} == "down"')
print("Label type:", LABEL_TYPE)
print("Future-facing alternative:", IDEAL_FUTURE_TARGET)
print("Forbidden proxy-leakage features:", LEAKAGE_COLUMNS)

Target: is_declining_label
Label source: trend_direction == "down"
Label type: Rule-derived observed-trend proxy
Future-facing alternative: Decline during a future time window, predicted from earlier data
Forbidden proxy-leakage features: ['trend_direction', 'trend_pct']


## 3. Success metric

My primary metric is Precision@20. It measures the proportion of the 20 highest-ranked eligible content items that have the positive decline proxy label. This fits the practical situation where an editor can review only a limited number of pages. I will compare it against a simple baseline and report the positive-label prevalence for context. The metric evaluates identification of the defined decline proxy, not the causal impact of refreshing content.

In [6]:
# Section 3: Define the success metric

PRIMARY_METRIC = "Precision@20"
K = 20
BASELINE = "Simple transparent ranking using permitted features"

def precision_at_k(y_true, scores, k=20):
    """
    Fraction of the top-k ranked items whose true label is positive.
    Higher is better. Labels must be binary (0/1).
    """
    y = pd.Series(y_true).reset_index(drop=True)
    s = pd.Series(scores).reset_index(drop=True)

    if len(y) != len(s):
        raise ValueError("Labels and scores must have equal length.")
    if len(y) == 0:
        raise ValueError("At least one item is required.")
    if not y.isin([0, 1]).all():
        raise ValueError("Labels must contain only 0 and 1.")

    k = min(k, len(y))
    top_indices = s.sort_values(ascending=False).head(k).index
    return float(y.loc[top_indices].mean())

print("Primary metric:", PRIMARY_METRIC)
print("Review capacity (K):", K)
print("Comparison baseline:", BASELINE)
print("Interpretation: positive labels among the top 20 / 20")

Primary metric: Precision@20
Review capacity (K): 20
Comparison baseline: Simple transparent ranking using permitted features
Interpretation: positive labels among the top 20 / 20


## 4. The unit of analysis, as a real dataframe

The unit of analysis is one pseudonymized content item, identified by content_id. Each row represents one page and its associated metadata and aggregated performance measurements. The table below shows real records and the proxy label, so I can inspect the target and data grain before selecting features. The raw dataset is read-only; I will not modify or commit a copy of it.

In [7]:
# Section 4: Locate and load the starter dataset

from pathlib import Path
import pandas as pd

cwd = Path.cwd()

candidate_roots = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
]

project_root = next(
    (
        root for root in candidate_roots
        if (root / "data" / "raw" / "content_refresh_anonymized.csv").exists()
    ),
    None,
)

if project_root is not None:
    data_path = (
        project_root / "data" / "raw" / "content_refresh_anonymized.csv"
    )
    df = pd.read_csv(data_path)
else:
    # Fallback for a Colab session that has not cloned the repository.
    from urllib.request import urlopen
    from io import BytesIO

    raw_url = (
        "https://raw.githubusercontent.com/"
        "mimicodegirl-26/flyrank-ml-internship/"
        "main/data/raw/content_refresh_anonymized.csv"
    )

    try:
        with urlopen(raw_url, timeout=30) as response:
            df = pd.read_csv(BytesIO(response.read()))
    except Exception as exc:
        raise FileNotFoundError(
            "Could not load the starter CSV. Clone your repository in Colab "
            "or ensure data/raw/content_refresh_anonymized.csv exists."
        ) from exc

required_columns = {
    "content_id",
    "trend_direction",
}

missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

# Reconstruct the documented starter proxy if necessary.
if "is_declining_label" not in df.columns:
    df["is_declining_label"] = (
        df["trend_direction"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("down")
        .astype("int8")
    )

print("Dataset shape:", df.shape)
print("Unique content items:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

display(df.head())

# Show a compact, real dataframe of content items and the target.
preferred_columns = [
    "content_id",
    "content_type",
    "content_age_days",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "trend_direction",
    "is_declining_label",
]

preview_columns = [c for c in preferred_columns if c in df.columns]
unit_df = df[preview_columns].head(10).copy()

display(unit_df)

print("Unit of analysis: one row = one pseudonymized content item.")

print("\nTarget distribution:")
display(
    df["is_declining_label"]
    .value_counts(dropna=False)
    .rename_axis("is_declining_label")
    .to_frame("content_item_count")
)

print("\nTarget prevalence:")
display(
    df["is_declining_label"]
    .value_counts(normalize=True, dropna=False)
    .rename_axis("is_declining_label")
    .to_frame("proportion")
)

Dataset shape: (30000, 45)
Unique content items: 30000
Duplicate content IDs: 0


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1


,content_id,content_type,content_age_days,impressions_90d,clicks_90d,sessions_90d,trend_direction,is_declining_label
0,content_304f48230142,keyword article,187,3803,29,17,down,1
1,content_a1fb4e703a9e,keyword article,445,15320,7,9,down,1
2,content_9aa793d4d895,keyword article,141,12581,11,11,down,1
3,content_331d6c4de07b,keyword article,463,11751,58,78,stable,0
4,content_d99b7a2d90ca,keyword article,263,19140,24,145,down,1
5,content_d4084a4bc775,keyword article,147,3970,1,5,down,1
6,content_9a34b442b552,keyword article,90,20,0,1,down,1
7,content_a63219c6e95a,keyword article,445,1724,1,28,stable,0
8,content_5e6c160719bc,keyword article,90,32574,29,68,down,1
9,content_c27558df2b0c,keyword article,257,1240,2,3,down,1


Unit of analysis: one row = one pseudonymized content item.

Target distribution:


,content_item_count
is_declining_label,
1,16262
0,13738



Target prevalence:


,proportion
is_declining_label,
1,0.542067
0,0.457933


## 5. Why ML beats a fixed rule here

Content performance is influenced by multiple signals, including impressions, clicks, content age, freshness, search position, and engagement. A single fixed threshold may miss pages where several moderate signals combine into a meaningful review opportunity. A model may learn these interactions and rank items more usefully, but that benefit must be tested rather than assumed. I will compare a transparent rule-based baseline with a learned model using the same eligible items and a leakage-safe evaluation. The final ranking will support human review, not make automatic publishing or editing decisions.

In [8]:
# Section 5: Inspect candidate signals and document leakage exclusions

IDENTIFIER_COLUMNS = ["content_id", "client_id"]
TARGET_COLUMNS = ["is_declining_label"]
LEAKAGE_COLUMNS = ["trend_direction", "trend_pct"]

# Potential predictors are discovered from the real dataframe.
# This is an inventory, not a finalized model feature list.
excluded_columns = set(
    IDENTIFIER_COLUMNS + TARGET_COLUMNS + LEAKAGE_COLUMNS
)

candidate_features = [
    col for col in df.columns
    if col not in excluded_columns
]

print("Available candidate predictors:", len(candidate_features))
print("\nExample candidate features:")
print(candidate_features[:20])

print("\nExcluded identifiers, target and leakage fields:")
print(sorted(excluded_columns.intersection(df.columns)))

assert "is_declining_label" not in candidate_features
assert "trend_direction" not in candidate_features
assert "trend_pct" not in candidate_features

# Verify that the target matches its documented rule.
expected_label = (
    df["trend_direction"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("down")
    .astype("int8")
)

label_matches = (
    df["is_declining_label"].astype("int8") == expected_label
)

print("\nRows matching the documented label rule:",
      int(label_matches.sum()), "/", len(df))
print("Label-rule match rate:", round(label_matches.mean(), 4))

# A fixed rule is a useful baseline, not something to dismiss.
print(
    "\nNext comparison: evaluate a transparent baseline and a learned "
    "ranking on the same held-out items using Precision@20."
)

Available candidate predictors: 40

Example candidate features:
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions']

Excluded identifiers, target and leakage fields:
['client_id', 'content_id', 'is_declining_label', 'trend_direction', 'trend_pct']

Rows matching the documented label rule: 30000 / 30000
Label-rule match rate: 1.0

Next comparison: evaluate a transparent baseline and a learned ranking on the same held-out items using Precision@20.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.